In [8]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

In [2]:
dataset = load_dataset("PolyAI/banking77")

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

label_names = dataset["train"].features["label"].names

X_train, X_val, y_train, y_val = train_test_split(
    train_df["text"],
    train_df["label"],
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"]
)

X_test = test_df["text"]
y_test = test_df["label"]

Found cached dataset banking77 (C:/Users/alaaj/.cache/huggingface/datasets/PolyAI___banking77/default/1.1.0/17ffc2ed47c2ed928bee64127ff1dbc97204cb974c2f980becae7c864007aed9)


  0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
print(len(X_train))
print(len(X_val))
print(len(X_test))

8002
2001
3080


In [5]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])

In [6]:
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('classifier', LogisticRegression(max_iter=1000))])

In [7]:
val_predictions = model.predict(X_val)

In [9]:
accuracy = accuracy_score(
    y_val,
    val_predictions
)

macro_f1 = f1_score(
    y_val,
    val_predictions,
    average="macro"
)

print("Validation Accuracy:", accuracy)
print("Validation Macro F1:", macro_f1)

Validation Accuracy: 0.8495752123938031
Validation Macro F1: 0.8418410951929223


| Model                        |    Accuracy |    Macro F1 |
| ---------------------------- | ----------: | ----------: |
| Dummy Most Frequent          | 0.018990504 | 0.000484067 |
| TF-IDF + Logistic Regression | 0.849575212 | 0.841841095 |
| TF-IDF + SVM                 |       later |       later |
| Transformer                  |  much later |  much later |


In [13]:
YOUR_DUMMY_ACCURACY = 0.018990504
YOUR_DUMMY_MACRO_F1 = 0.000484067

In [14]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Dummy Most Frequent",
        "TF-IDF + Logistic Regression"
    ],
    "Accuracy": [
        YOUR_DUMMY_ACCURACY,
        accuracy
    ],
    "Macro F1": [
        YOUR_DUMMY_MACRO_F1,
        macro_f1
    ]
})

results

,Model,Accuracy,Macro F1
0,Dummy Most Frequent,0.018991,0.841841
1,TF-IDF + Logistic Regression,0.849575,0.841841


TF-IDF + Logistic Regression is 83% better 

In [24]:
report_dict = classification_report(
    y_val,
    val_predictions,
    target_names=label_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).T

In [29]:
class_results = report_df.loc[label_names]

In [30]:
class_results.sort_values(
    "f1-score",
    ascending=False
).head(5)

,precision,recall,f1-score,support
apple_pay_or_google_pay,0.961538,1.000000,0.980392,25.0
lost_or_stolen_phone,1.000000,0.958333,0.978723,24.0
visa_or_mastercard,0.962963,0.962963,0.962963,27.0
passcode_forgotten,0.952381,0.952381,0.952381,21.0
request_refund,0.916667,0.970588,0.942857,34.0


In [31]:
class_results.sort_values(
    "f1-score"
).head(5)

,precision,recall,f1-score,support
virtual_card_not_working,1.000000,0.250000,0.400000,8.0
pin_blocked,0.823529,0.608696,0.700000,23.0
pending_transfer,0.687500,0.733333,0.709677,30.0
extra_charge_on_statement,0.666667,0.787879,0.722222,33.0
card_acceptance,0.800000,0.666667,0.727273,12.0


In [32]:
errors = pd.DataFrame({
    "text": X_val.values,
    "true_label": y_val.values,
    "predicted_label": val_predictions
})

In [33]:
errors["true_intent"] = errors["true_label"].apply(
    lambda x: label_names[x]
)

errors["predicted_intent"] = errors["predicted_label"].apply(
    lambda x: label_names[x]
)

In [34]:
errors = errors[
    errors["true_label"] != errors["predicted_label"]
]

In [35]:
errors[
    ["text", "true_intent", "predicted_intent"]
].sample(20, random_state=42)

,text,true_intent,predicted_intent
1190,Where did the money transfer go?,transfer_not_received_by_recipient,failed_transfer
1938,How many payments can I make using a virtual c...,disposable_card_limits,getting_virtual_card
1515,My transfer seems to be too expensive.,transfer_fee_charged,failed_transfer
1372,Is there a place to check for fee for payments...,card_payment_fee_charged,cash_withdrawal_charge
374,Are you going to charge me to collect the money?,top_up_by_bank_transfer_charge,top_up_by_card_charge
54,i tried to do a transfer to an account but it ...,beneficiary_not_allowed,failed_transfer
718,I noticed that there was an extra charge fee o...,transfer_fee_charged,extra_charge_on_statement
995,Someone that I had to send money to told me th...,transfer_fee_charged,receiving_money
1072,Why was my account assessed a fee?,extra_charge_on_statement,transfer_fee_charged
205,What is the time period for money to be transf...,transfer_timing,balance_not_updated_after_bank_transfer


In [36]:
confusion_pairs = (
    errors.groupby(
        ["true_intent", "predicted_intent"]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

confusion_pairs.head(15)

,true_intent,predicted_intent,count
133,pin_blocked,get_physical_card,8
169,transfer_fee_charged,extra_charge_on_statement,6
196,virtual_card_not_working,getting_virtual_card,5
90,exchange_rate,card_payment_wrong_exchange_rate,5
209,wrong_exchange_rate_for_cash_withdrawal,wrong_amount_of_cash_received,4
127,pending_top_up,top_up_failed,4
187,unable_to_verify_identity,why_verify_identity,4
180,transfer_not_received_by_recipient,pending_transfer,4
132,pending_transfer,transfer_timing,3
128,pending_top_up,top_up_reverted,3


In [39]:
pair_errors = errors[
    (errors["true_intent"] == "pin_blocked") &
    (errors["predicted_intent"] == "get_physical_card")
]

pair_errors[
    ["text", "true_intent", "predicted_intent"]
].head(10)

,text,true_intent,predicted_intent
244,I cannot use my PIN.,pin_blocked,get_physical_card
408,When I was drunk I got my card stuck in your m...,pin_blocked,get_physical_card
499,I can't input my pin again.,pin_blocked,get_physical_card
1183,How do I make my PIN work again?,pin_blocked,get_physical_card
1626,How do I go about unlocking a pin?,pin_blocked,get_physical_card
1631,"While I was drunk, I goofed my PIN number and ...",pin_blocked,get_physical_card
1977,What is the process for unlocking the pin?,pin_blocked,get_physical_card
1986,"My PIN can't access my card, can you help?.",pin_blocked,get_physical_card


In [41]:
pair_errors = errors[
    (errors["true_intent"] == "transfer_fee_charged") &
    (errors["predicted_intent"] == "extra_charge_on_statement")
]

pair_errors[
    ["text", "true_intent", "predicted_intent"]
].head(10)

,text,true_intent,predicted_intent
380,Can you elaborate on why I have an extra charg...,transfer_fee_charged,extra_charge_on_statement
718,I noticed that there was an extra charge fee o...,transfer_fee_charged,extra_charge_on_statement
843,How come I can see an extra charge for my tran...,transfer_fee_charged,extra_charge_on_statement
1079,I didn't know there was a charge for tranferri...,transfer_fee_charged,extra_charge_on_statement
1407,I got an unexpected charge on my transfers. Ho...,transfer_fee_charged,extra_charge_on_statement
1808,Why am I seeing a transfer fee on my bank stat...,transfer_fee_charged,extra_charge_on_statement


In [40]:
pair_errors = errors[
    (errors["true_intent"] == "exchange_rate") &
    (errors["predicted_intent"] == "card_payment_wrong_exchange_rate")
]

pair_errors[
    ["text", "true_intent", "predicted_intent"]
].head(10)

,text,true_intent,predicted_intent
418,Is the exchange rate the same on weekends as t...,exchange_rate,card_payment_wrong_exchange_rate
869,explain the interbank exchange rate,exchange_rate,card_payment_wrong_exchange_rate
1385,Is there a specific source that the exchange r...,exchange_rate,card_payment_wrong_exchange_rate
1447,What is the system for determining the exchang...,exchange_rate,card_payment_wrong_exchange_rate
1827,Where can I find the exchange rate for my tran...,exchange_rate,card_payment_wrong_exchange_rate


In [42]:
tfidf = model.named_steps["tfidf"]

feature_names = tfidf.get_feature_names_out()

print("Number of TF-IDF features:", len(feature_names))

Number of TF-IDF features: 2156


In [43]:
feature_names[:30]

array(['00', '000', '10', '100', '13', '18', '1818', '1l', '20', '200',
       '2018', '30', '3d', '40', '45', '50', '500', '5x', '60', '80',
       'able', 'about', 'above', 'abroad', 'absolutely', 'accept',
       'acceptable', 'accepted', 'accepting', 'accepts'], dtype=object)

In [44]:
sample_matrix = tfidf.transform(
    X_val.iloc[:5]
)

sample_matrix.shape

(5, 2156)

In [45]:
type(sample_matrix)

scipy.sparse.csr.csr_matrix

In [46]:
probabilities = model.predict_proba(X_val)

In [47]:
probabilities.shape

(2001, 77)

In [48]:
max_confidence = probabilities.max(axis=1)

In [49]:
prediction_analysis = pd.DataFrame({
    "text": X_val.values,
    "true_label": y_val.values,
    "predicted_label": val_predictions,
    "confidence": max_confidence
})

In [50]:
prediction_analysis.sort_values(
    "confidence"
).head(20)

,text,true_label,predicted_label,confidence
634,What is the first step I need to make for bein...,74,1,0.038868
299,Who accepts the card?,10,24,0.039259
981,The card is non-functional.,14,11,0.041874
921,"Needed gas, been delayed for half an hour.Plea...",47,47,0.043116
1765,I'm having issues with my card. You guys keep ...,27,25,0.043121
881,i havent got my card,11,11,0.045080
1533,I'm interested in getting a card. How does one...,43,24,0.045393
1908,"Is it possible to go ahead and log in, althoug...",74,66,0.046419
534,How do I actually obtain a card myself?,43,43,0.046429
1378,Is there a way to get around fees,15,15,0.047059


In [52]:
TfidfVectorizer(
    ngram_range=(1, 2)
)

TfidfVectorizer(ngram_range=(1, 2))

In [59]:
bigram_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2)
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])


In [60]:
bigram_model.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
                ('classifier', LogisticRegression(max_iter=1000))])

In [61]:
val_predictions2 = bigram_model.predict(X_val)

In [62]:
accuracy = accuracy_score(
    y_val,
    val_predictions2
)

macro_f1 = f1_score(
    y_val,
    val_predictions2,
    average="macro"
)

print("Validation Accuracy:", accuracy)
print("Validation Macro F1:", macro_f1)

Validation Accuracy: 0.8415792103948025
Validation Macro F1: 0.8296295497892721


| Model                                      |    Accuracy |    Macro F1 |
| ------------------------------------------ | ----------: | ----------: |
| Dummy Most Frequent                        | 0.018990504 | 0.000484067 |
|TF-IDF unigram + Logistic Regression        | 0.849575212 | 0.841841095 |
|TF-IDF unigram+bigram + Logistic Regression | 0.841579210 | 0.829629549 |



In [ ]:
Q1 

it gives less importance to words that appear in many documents and more importance to words that are more distinctive.

Q2

because 



